In [6]:
import pandas as pd
import altair as alt
import colorsys

# 1. Cargar el archivo con el nombre correcto (usando espacios en lugar de guiones bajos)
df = pd.read_excel("/content/BASE DE DATOS FINAL - PAISES SERPARADOS.xlsx", sheet_name=0)

# 2. Una fila por película (evita contar varias veces la misma película)
peliculas = df.drop_duplicates(
    subset=["titulo_original_pelicula_nominada", "ano_pelicula_nominada"]
)

# 3. Conteo de películas por país de producción
conteo = peliculas["paises_de_produccion"].value_counts().reset_index()
conteo.columns = ["pais", "peliculas"]

# 4. Paleta con un color distinto por cada país
n = len(conteo)
colores = [
    "#%02x%02x%02x" % tuple(int(c * 255) for c in colorsys.hsv_to_rgb((i * 0.618) % 1, 0.65, 0.9))
    for i in range(n)
]
orden = conteo["pais"].tolist()

# 5. Gráfico de barras horizontales
barras = alt.Chart(conteo).mark_bar().encode(
    y=alt.Y("pais:N", sort=orden, title="País de producción"),
    x=alt.X("peliculas:Q", title="Cantidad de películas nominadas"),
    color=alt.Color(
        "pais:N",
        scale=alt.Scale(domain=orden, range=colores),
        legend=None,
    ),
    tooltip=[
        alt.Tooltip("pais:N", title="País"),
        alt.Tooltip("peliculas:Q", title="Películas"),
    ],
)

# Etiquetas con el valor al final de cada barra
etiquetas = alt.Chart(conteo).mark_text(align="left", dx=4).encode(
    y=alt.Y("pais:N", sort=orden),
    x="peliculas:Q",
    text="peliculas:Q",
)

grafico = (barras + etiquetas).properties(
    width=700,
    height=550,
    title="País de producción que más predomina en las películas nominadas",
)

grafico

alt.LayerChart(...)

In [9]:
import ipywidgets as widgets
from IPython.display import display, clear_output

def grafico_anio(anio):
    datos = tabla[tabla["anio"] == anio]
    barras = alt.Chart(datos).mark_bar().encode(
        y=alt.Y("pais:N", sort="-x", title="País de producción"),
        x=alt.X("peliculas:Q", title="Películas nominadas (acumulado)",
                scale=alt.Scale(domain=[0, max_total])),
        color=alt.Color("pais:N", scale=alt.Scale(domain=top_paises, range=colores), legend=None),
        tooltip=["pais:N", "peliculas:Q"],
    )
    etiquetas = alt.Chart(datos).mark_text(align="left", dx=4).encode(
        y=alt.Y("pais:N", sort="-x"), x="peliculas:Q", text="peliculas:Q"
    )
    return (barras + etiquetas).properties(
        width=650, height=400, title=f"Películas nominadas acumuladas hasta {anio}"
    )

play = widgets.Play(value=anios[0], min=anios[0], max=anios[-1], step=2, interval=500)
slider_w = widgets.IntSlider(value=anios[0], min=anios[0], max=anios[-1], description="Año")
widgets.jslink((play, "value"), (slider_w, "value"))
salida = widgets.Output()

def actualizar(cambio):
    with salida:
        clear_output(wait=True)
        display(grafico_anio(cambio["new"]))

slider_w.observe(actualizar, names="value")

display(widgets.HBox([play, slider_w]), salida)
actualizar({"new": anios[0]})

Output()

In [12]:
import pandas as pd
import altair as alt
import colorsys

# 1. Cargar la primera hoja
df = pd.read_excel("BASE DE DATOS FINAL - PAISES SERPARADOS.xlsx", sheet_name=0)

# 2. Una fila por película (evita contar varias veces la misma película)
peliculas = df.drop_duplicates(
    subset=["titulo_original_pelicula_nominada", "ano_pelicula_nominada"]
)

# 3. Columnas de país de producción
cols_paises = ["paises_de_produccion"] + [f"pais_de_prodduccion_{i}" for i in range(2, 9)]

# 4. Juntar todas las columnas en una sola, quitar vacíos y contar
paises = (
    peliculas[cols_paises]
    .stack()                      # descarta automáticamente los NaN
    .astype(str).str.strip()
)
paises = paises[paises != ""]

conteo = paises.value_counts().reset_index()
conteo.columns = ["pais", "peliculas"]

# 5. Un color distinto por país
n = len(conteo)
colores = [
    "#%02x%02x%02x" % tuple(int(c * 255) for c in colorsys.hsv_to_rgb((i * 0.618) % 1, 0.65, 0.9))
    for i in range(n)
]
orden = conteo["pais"].tolist()

# 6. Gráfico de barras horizontales
barras = alt.Chart(conteo).mark_bar().encode(
    y=alt.Y("pais:N", sort=orden, title="País de producción"),
    x=alt.X("peliculas:Q", title="Cantidad de películas nominadas"),
    color=alt.Color("pais:N", scale=alt.Scale(domain=orden, range=colores), legend=None),
    tooltip=[alt.Tooltip("pais:N", title="País"),
             alt.Tooltip("peliculas:Q", title="Películas")],
)

etiquetas = alt.Chart(conteo).mark_text(align="left", dx=4).encode(
    y=alt.Y("pais:N", sort=orden),
    x="peliculas:Q",
    text="peliculas:Q",
)

grafico = (barras + etiquetas).properties(
    width=700,
    height=n * 22,   # la altura se ajusta a la cantidad de países
    title="Países de producción en las películas nominadas (todas las columnas de país)",
)

grafico

alt.LayerChart(...)

In [13]:
import pandas as pd
import altair as alt

# 1. Cargar la primera hoja
df = pd.read_excel("BASE DE DATOS FINAL - PAISES SERPARADOS.xlsx", sheet_name=0)
df["decada"] = (df["ano_ceremonia"] // 10) * 10

# 2. Indicador de actores: % de nominados/as no estadounidenses
#    (strip() porque hay valores con espacios sobrantes, ej. "Estadounidense ")
df["nac"] = df["nacionalidad_1"].astype(str).str.strip()
actores = (
    df.assign(v=df["nac"] != "Estadounidense")
    .groupby("decada")["v"].mean().mul(100).reset_index(name="valor")
)
actores["indicador"] = "Nominados/as no estadounidenses"

# 3. Indicadores de películas (una fila por película, todas las columnas de país)
cols_paises = ["paises_de_produccion"] + [f"pais_de_prodduccion_{i}" for i in range(2, 9)]
peliculas = df.drop_duplicates(
    subset=["titulo_original_pelicula_nominada", "ano_pelicula_nominada"]
).copy()
peliculas["n_paises"] = peliculas[cols_paises].notna().sum(axis=1)
peliculas["con_us"] = peliculas[cols_paises].eq("Estados Unidos").any(axis=1)

sin_us = (
    peliculas.assign(v=~peliculas["con_us"])
    .groupby("decada")["v"].mean().mul(100).reset_index(name="valor")
)
sin_us["indicador"] = "Películas sin participación de EE. UU."

coprod = (
    peliculas.assign(v=peliculas["n_paises"] > 1)
    .groupby("decada")["v"].mean().mul(100).reset_index(name="valor")
)
coprod["indicador"] = "Películas coproducidas (2+ países)"

# 4. Unir todo en una sola tabla
datos = pd.concat([actores, sin_us, coprod], ignore_index=True)
datos["decada_txt"] = datos["decada"].astype(str) + "s"
datos["valor"] = datos["valor"].round(1)

# 5. Gráfico de líneas (clic en la leyenda para resaltar un indicador)
seleccion = alt.selection_point(fields=["indicador"], bind="legend")

grafico = (
    alt.Chart(datos)
    .mark_line(point=alt.OverlayMarkDef(size=70), strokeWidth=3)
    .encode(
        x=alt.X("decada_txt:O", title="Década de la ceremonia"),
        y=alt.Y("valor:Q", title="Porcentaje (%)", scale=alt.Scale(domain=[0, 100])),
        color=alt.Color(
            "indicador:N",
            title="Indicador",
            scale=alt.Scale(
                domain=[
                    "Nominados/as no estadounidenses",
                    "Películas sin participación de EE. UU.",
                    "Películas coproducidas (2+ países)",
                ],
                range=["#e4572e", "#2e86ab", "#76b041"],
            ),
        ),
        opacity=alt.condition(seleccion, alt.value(1), alt.value(0.15)),
        tooltip=[
            alt.Tooltip("decada_txt:O", title="Década"),
            alt.Tooltip("indicador:N", title="Indicador"),
            alt.Tooltip("valor:Q", title="Porcentaje (%)"),
        ],
    )
    .add_params(seleccion)
    .properties(
        width=700,
        height=400,
        title="Internacionalización de las nominaciones al Oscar a lo largo del tiempo",
    )
)

grafico

alt.Chart(...)

In [17]:
# Guardar el gráfico interactivo como un archivo HTML y descargarlo
import altair as alt
from google.colab import files

# 1. Definir el nombre del archivo
nombre_archivo = "internacionalizacion_oscars.html"

# 2. Guardar el objeto grafico interactivo en formato HTML
grafico.save(nombre_archivo)

print(f"Archivo '{nombre_archivo}' guardado con éxito.")

# 3. Descargar el archivo directamente a tu computadora
files.download(nombre_archivo)

Archivo 'internacionalizacion_oscars.html' guardado con éxito.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
import pandas as pd
import altair as alt

# 1. Cargar la primera hoja
df = pd.read_excel("BASE DE DATOS FINAL - PAISES SERPARADOS.xlsx", sheet_name=0)
df["decada"] = (df["ano_ceremonia"] // 10) * 10

# 2. Indicador de actores: % de nominados/as no estadounidenses
#    (strip() porque hay valores con espacios sobrantes, ej. "Estadounidense ")
df["nac"] = df["nacionalidad_1"].astype(str).str.strip()
actores = (
    df.assign(v=df["nac"] != "Estadounidense")
    .groupby("decada")["v"].mean().mul(100).reset_index(name="valor")
)
actores["indicador"] = "Nominados/as no estadounidenses"

# 3. Indicadores de películas (una fila por película, todas las columnas de país)
cols_paises = ["paises_de_produccion"] + [f"pais_de_prodduccion_{i}" for i in range(2, 9)]
peliculas = df.drop_duplicates(
    subset=["titulo_original_pelicula_nominada", "ano_pelicula_nominada"]
).copy()
peliculas["n_paises"] = peliculas[cols_paises].notna().sum(axis=1)
peliculas["con_us"] = peliculas[cols_paises].eq("Estados Unidos").any(axis=1)

def porcentaje_por_decada(condicion, nombre):
    out = (
        peliculas.assign(v=condicion)
        .groupby("decada")["v"].mean().mul(100).reset_index(name="valor")
    )
    out["indicador"] = nombre
    return out

solo_us = porcentaje_por_decada(
    peliculas["con_us"] & (peliculas["n_paises"] == 1),
    "Películas solo estadounidenses",
)
sin_us = porcentaje_por_decada(
    ~peliculas["con_us"],
    "Películas sin participación de EE. UU.",
)
coprod = porcentaje_por_decada(
    peliculas["n_paises"] > 1,
    "Películas coproducidas (2+ países)",
)

# 4. Unir todo en una sola tabla
datos = pd.concat([actores, solo_us, sin_us, coprod], ignore_index=True)
datos["decada_txt"] = datos["decada"].astype(str) + "s"
datos["valor"] = datos["valor"].round(1)

# 5. Gráfico de líneas (clic en la leyenda para resaltar un indicador)
orden_indicadores = [
    "Nominados/as no estadounidenses",
    "Películas solo estadounidenses",
    "Películas sin participación de EE. UU.",
    "Películas coproducidas (2+ países)",
]
paleta = ["#e4572e", "#1d3557", "#2e86ab", "#76b041"]

seleccion = alt.selection_point(fields=["indicador"], bind="legend")

grafico = (
    alt.Chart(datos)
    .mark_line(point=alt.OverlayMarkDef(size=70), strokeWidth=3)
    .encode(
        x=alt.X("decada_txt:O", title="Década de la ceremonia"),
        y=alt.Y("valor:Q", title="Porcentaje (%)", scale=alt.Scale(domain=[0, 100])),
        color=alt.Color(
            "indicador:N",
            title="Indicador",
            scale=alt.Scale(domain=orden_indicadores, range=paleta),
        ),
        opacity=alt.condition(seleccion, alt.value(1), alt.value(0.15)),
        tooltip=[
            alt.Tooltip("decada_txt:O", title="Década"),
            alt.Tooltip("indicador:N", title="Indicador"),
            alt.Tooltip("valor:Q", title="Porcentaje (%)"),
        ]
    )
    .add_params(seleccion)
    .properties(
        width=700,
        height=400,
        title="Internacionalización de las nominaciones al Oscar a lo largo del tiempo",
    )
)

grafico

alt.Chart(...)

In [10]:
import pandas as pd
import altair as alt

# 1. Cargar la primera hoja
df = pd.read_excel("BASE DE DATOS FINAL - PAISES SERPARADOS.xlsx", sheet_name=0)
df["decada"] = (df["ano_ceremonia"] // 10) * 10

# 2. Una fila por película, usando todas las columnas de país de producción
cols_paises = ["paises_de_produccion"] + [f"pais_de_prodduccion_{i}" for i in range(2, 9)]
peliculas = df.drop_duplicates(
    subset=["titulo_original_pelicula_nominada", "ano_pelicula_nominada"]
).copy()
peliculas["n_paises"] = peliculas[cols_paises].notna().sum(axis=1)
peliculas["con_us"] = peliculas[cols_paises].eq("Estados Unidos").any(axis=1)

def porcentaje_peliculas(condicion, nombre):
    out = (
        peliculas.assign(v=condicion)
        .groupby("decada")["v"].mean().mul(100).reset_index(name="valor")
    )
    out["indicador"] = nombre
    return out

# 3. Indicadores
solo_us = porcentaje_peliculas(
    peliculas["con_us"] & (peliculas["n_paises"] == 1),
    "Películas solo estadounidenses",
)
extranjeras = porcentaje_peliculas(
    ~peliculas["con_us"],
    "Películas extranjeras",
)
coprod = porcentaje_peliculas(
    peliculas["n_paises"] > 1,
    "Películas coproducidas (2+ países)",
)

# 4. Unir todo en una sola tabla
datos = pd.concat([solo_us, extranjeras, coprod], ignore_index=True)
datos["decada_txt"] = datos["decada"].astype(str) + "s"
datos["valor"] = datos["valor"].round(1)

# 5. Gráfico de líneas (clic en la leyenda para resaltar un indicador)
orden_indicadores = [
    "Películas solo estadounidenses",
    "Películas extranjeras",
    "Películas coproducidas (2+ países)",
]
paleta = ["#670010", "#C69D22", "#181818"]

seleccion = alt.selection_point(fields=["indicador"], bind="legend")

grafico = (
    alt.Chart(datos)
    .mark_line(point=alt.OverlayMarkDef(size=70), strokeWidth=3)
    .encode(
        x=alt.X("decada_txt:O", title="Década de la ceremonia"),
        y=alt.Y("valor:Q", title="Porcentaje (%)", scale=alt.Scale(domain=[0, 100])),
        color=alt.Color(
            "indicador:N",
            title="Indicador",
            scale=alt.Scale(domain=orden_indicadores, range=paleta),
        ),
        opacity=alt.condition(seleccion, alt.value(1), alt.value(0.15)),
        tooltip=[
            alt.Tooltip("decada_txt:O", title="Década"),
            alt.Tooltip("indicador:N", title="Indicador"),
            alt.Tooltip("valor:Q", title="Porcentaje (%)"),
        ],
    )
    .add_params(seleccion)
    .properties(
        width=700,
        height=400,
        title="Internacionalización de las nominaciones al Oscar a lo largo del tiempo",
    )
)

grafico

alt.Chart(...)

In [11]:
# Guardar el gráfico interactivo actualizado con los nuevos colores como HTML y descargarlo
from google.colab import files

nombre_archivo_actualizado = "internacionalizacion_oscars_actualizado.html"

# Guardar el objeto grafico interactivo que se generó en la celda anterior
grafico.save(nombre_archivo_actualizado)

print(f"Archivo '{nombre_archivo_actualizado}' guardado con éxito con la nueva paleta de colores.")

# Descargar el archivo
files.download(nombre_archivo_actualizado)

Archivo 'internacionalizacion_oscars_actualizado.html' guardado con éxito con la nueva paleta de colores.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>